In [1]:
import json

def write_jsonl(data, file_path):
    """
    Write a list of dictionaries to a JSONL file.

    :param data: List of dictionaries to write to the file
    :param file_path: Path to the JSONL file
    """
    with open(file_path, 'w') as file:
        for entry in data:
            json_line = json.dumps(entry)
            file.write(json_line + '\n')

def read_jsonl(file_path):
    """
    Read a JSONL file and return a list of dictionaries.

    :param file_path: Path to the JSONL file
    :return: List of dictionaries
    """
    with open(file_path, 'r') as file:
        return [json.loads(line) for line in file]

import csv
def read_tsv(file_path):
    with open(file_path, 'r') as file:
        return [line for line in csv.reader(file, delimiter='\t')]
    
def write_tsv(data, file_path):
    with open(file_path, 'w') as file:
        writer = csv.writer(file, delimiter='\t')
        writer.writerows(data)
        
def read_json(file_path):
    with open(file_path, 'r') as file:
        return json.load(file)

In [2]:
from pathlib import Path
import csv
import sys
maxInt = sys.maxsize

while True:
    # decrease the maxInt value by factor 10 
    # as long as the OverflowError occurs.

    try:
        csv.field_size_limit(maxInt)
        break
    except OverflowError:
        maxInt = int(maxInt/10)
        
def load_hf(object_class, model_name):
    try:
        obj = object_class.from_pretrained(model_name, local_files_only=True)
    except:
        obj = object_class.from_pretrained(model_name, local_files_only=False)
    return obj


In [3]:

import transformers

tokenizer = load_hf(transformers.AutoTokenizer, 'facebook/contriever-msmarco')
data_type='qampari'
split='dev'


/scratch/cluster/hungting/projects/diverse_response/div/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Map Proof Texts in the Data to Actual Documents in the Corpus

In [3]:
from pathlib import Path
from tqdm import tqdm

split='dev'
rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/data/qampari_data')
aligned_data_list = (rootdir / 'aligned_data' / split).glob('chunk_0_*.json')

q2docs = {}
for aligned_data_path in aligned_data_list:
    aligned_data = read_json(aligned_data_path)
    for inst in aligned_data:
        assert len(inst['ctxs']) == 1, (inst['question'], inst['ctxs'])
        q2docs[inst['question']] = inst['ctxs'][0]

data = read_jsonl(rootdir / f'{split}_data.jsonl')

proof_not_one = 0
for inst in tqdm(data):
    inst['ground_truths'] = [] 
    for ans in inst['answer_list']:  # for each ground truth answer
        if split == 'dev':  # for dev set, we have multiple proofs
            inst['ground_truths'].append([])
            for p in ans['proof']:
                inst['ground_truths'][-1].append(q2docs[p['proof_text']])
        else:
            proof_text = ans['proof'][0]['proof_text']
            inst['ground_truths'].append(q2docs[proof_text])
        
        if len(ans['proof']) != 1:
            proof_not_one += 1
        

print(proof_not_one)
write_jsonl(data, rootdir / f'{split}_data_gt_qampari_corpus.jsonl')

NameError: name 'read_json' is not defined

#### Count the number of positive contexts

In [ ]:
rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/data/qampari_data')

def count_len_ground_truths(data):
    len_ground_truths = 0
    for inst in data:
        len_ground_truths += len(inst['ground_truths'])
    return len_ground_truths / float(len(data))

split='dev'
data = read_jsonl(rootdir / 'old' / f'{split}_data_gt_qampari_corpus.jsonl')
print(count_len_ground_truths(data))
data = read_jsonl(rootdir / f'{split}_data_gt_qampari_corpus.jsonl')
print(count_len_ground_truths(data))

14.067581837381203
14.434


In [4]:

"""
    Generating training data for fine-tuning the 2nd stage model. 
    The training data consists of the following:
        - rewritten question: Question + top N ground truth documents (input documents)
        - positive documents: documents that are in the ground truth but not in the input documents
        - negative documents: Only consider random documents
        - hard negative documents: documents that are in the input documents
        
    Also, check if the rewritten question does not exceed 512 tokens. Omit the instance if it does.
"""

"""
    Combinations
    1. random
    2. random + input
    3. random + cont (hard)
    5. random + cont (hard) + input (hard)
    6. random + nv (hard)
    7. random + nv (hard) + input (hard)
"""
from tqdm import tqdm
import random

split='train'
corpus='qampari'
mode='random'

if split == 'train':
    train_str = '_train_gt' if split == 'train' else ''

    # # get top N retrieved documents
    rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/')

    # get ground truth documents
    data_path = rootdir / f'data/{data_type}_data/{split}_data_gt_{corpus}_corpus.jsonl'
    data = read_jsonl(data_path)

    i = 0        
    for inst in tqdm(data):
        # get the negative documents
        inst['negative_ctxs'] = []
        # get the random negative documents
        for _ in range(5):
            random_inst = random.choice(data)
            while random_inst['question_text'] == inst['question_text']:
                random_inst = random.choice(data)
            inst['negative_ctxs'] += random_inst['ground_truths'][:5]
        inst['hard_negative_ctxs'] = []
        inst['positive_ctxs'] = inst['ground_truths']

        
    print(len(data))
    len_eval_data = len(data) // 10
    new_eval_data = data[:len_eval_data]
    new_data = data[len_eval_data:]

    # write the new ground truth to a jsonl file
    write_jsonl(new_data, rootdir / f'training/data/{data_type}_data/{corpus}_corpus/{split}_data.jsonl')
    write_jsonl(new_eval_data, rootdir / f'training/data/{data_type}_data/{corpus}_corpus/{split}_eval_data.jsonl')


100%|██████████| 61911/61911 [00:00<00:00, 116464.13it/s]


61911


### Get input data with random negative / nv negatives / input negatives

In [ ]:

"""
    Generating training data for fine-tuning the 2nd stage model. 
    The training data consists of the following:
        - rewritten question: Question + top N ground truth documents (input documents)
        - positive documents: documents that are in the ground truth but not in the input documents
        - negative documents: Only consider random documents
        - hard negative documents: documents that are in the input documents
        
    Also, check if the rewritten question does not exceed 512 tokens. Omit the instance if it does.
"""

"""
    Combinations
    1. random
    2. random + input
    3. random + cont (hard)
    5. random + cont (hard) + input (hard)
    6. random + nv (hard)
    7. random + nv (hard) + input (hard)
"""
from tqdm import tqdm
import random

split='train'
corpus='qampari'
mode='random'

# for mode in ['random+input', 'random', 'random+cont', 'random+cont+input', 'random+nv', 'random+nv+input']:
for mode in ['random+input']:
    if split == 'train':
        train_str = '_train_gt' if split == 'train' else ''

        # # get top N retrieved documents
        rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/')

        # get ground truth documents
        data_path = rootdir / f'data/{data_type}_data/{split}_data_gt_{corpus}_corpus.jsonl'
        data = read_jsonl(data_path)
        
        
        # load contriever and NV docs (hard negatives)
        contriever_top_docs = read_jsonl("/scratch/cluster/hungting/projects/Multi_Answer/contriever/outputs/contriever_msmarco_qampari/qampari_train_random_data_qampari_corpus_finetuned_qampari_only_random.json") + read_jsonl("/scratch/cluster/hungting/projects/Multi_Answer/contriever/outputs/contriever_msmarco_qampari/qampari_train_eval_random_data_qampari_corpus_finetuned_qampari_only_random.json")
        # load contriever docs from data
        q2contdocs = {}
        for docs in contriever_top_docs:
            q2contdocs[docs['question'].split('\n\n')[0].split('Question: ')[1]] = docs['ctxs']
            
        # nv_top_docs = read_jsonl("/scratch/cluster/hungting/projects/Multi_Answer/mteb_retriever/outputs/nv-embed/2nd_stage_data/train_random_data_qampari_corpus.json") + read_jsonl("/scratch/cluster/hungting/projects/Multi_Answer/mteb_retriever/outputs/nv-embed/2nd_stage_data/train_eval_random_data_qampari_corpus.json")
        nv_top_docs = read_jsonl("/scratch/cluster/hungting/projects/Multi_Answer/mteb_retriever/outputs/nv-embed/qampari_train.json")
        # load nv docs from data
        q2nvdocs = {}
        for docs in nv_top_docs:
            q2nvdocs[docs['question'].split('\n\n')[0].split('Question: ')[1]] = docs['ctxs']

            
        # compute the difference 
        # identify the documents that are in the ground truth but not in the retrieved documents
        # these are the new ground truth
        TOP_N = 3
        for TOP_N in [0, 1, 2, 3]:
            new_data = []
            new_eval_data = []
            i = 0        
            for inst in tqdm(data):
                if len(inst['ground_truths']) <= TOP_N:  # ignore instances with less than 3 ground truth documents
                    continue
                
                # # filter out questions that contriever and nv don't retrieve
                # retrieved_documents = [doc for doc in inst['ground_truths'][:3]]
                # rewritten_question = 'Question: [Question]\n\nDocuments: [Documents]'.replace('[Question]', inst['question_text']).replace('[Documents]', '\n'.join([ddd['title'] + ' ' + ddd['text'] for ddd in retrieved_documents]))  
                # # check the rewritten question does not exceed 512 tokens
                # if len(tokenizer(rewritten_question)['input_ids']) > 512:
                #     rewritten_question = 'Question: [Question]\n\nDocuments: [Documents]'.replace('[Question]', inst['question_text']).replace('[Documents]', '\n'.join([ddd['title'] + ' ' + ddd['text'][:-50] for ddd in retrieved_documents]))  
                #     if len(tokenizer(rewritten_question)['input_ids']) > 512:
                #         print('Exceed 512 tokens', len(tokenizer(rewritten_question)['input_ids']))
                #         continue
                        
                # # get the hard negatives mined by contriever
                # contriever_top_doc = q2contdocs[inst['question_text']]
                
                # # # get the hard negatives mined by nv
                # nv_top_doc = q2nvdocs[inst['question_text']]
                    
                    
                from itertools import combinations
                # loop over different combinations of the top N retrieved documents
                
                num_combo = 0
                for combo in combinations(inst['ground_truths'], TOP_N):  # 2 for pairs, 3 for triplets, etc
                    if num_combo > 10:
                        break
                    # get the input documents
                    retrieved_documents = [doc for doc in combo]
                    if TOP_N == 0:
                        rewritten_question = inst['question_text']
                    else:
                        rewritten_question = 'Question: [Question]\n\nDocuments: [Documents]'.replace('[Question]', inst['question_text']).replace('[Documents]', '\n'.join([ddd['title'] + ' ' + ddd['text'] for ddd in retrieved_documents]))  
                        # check the rewritten question does not exceed 512 tokens
                        if len(tokenizer(rewritten_question)['input_ids']) > 512:
                            rewritten_question = 'Question: [Question]\n\nDocuments: [Documents]'.replace('[Question]', inst['question_text']).replace('[Documents]', '\n'.join([ddd['title'] + ' ' + ddd['text'][:-50] for ddd in retrieved_documents]))  
                            if len(tokenizer(rewritten_question)['input_ids']) > 512:
                                print('Exceed 512 tokens', len(tokenizer(rewritten_question)['input_ids']))
                                continue
                    
                    new_inst = {'id': i, 'question': rewritten_question, 'org_q': inst['question_text']}
                    

                    # get the positive documents
                    new_inst['positive_ctxs'] = []
                    for doc in inst['ground_truths']:
                        if doc not in retrieved_documents:
                            new_inst['positive_ctxs'].append(doc)
                    if len(new_inst['positive_ctxs']) == 0:
                        continue
                    new_ground_truth_titles = [doc['title'] for doc in new_inst['positive_ctxs']]    
                    
                    # get the negative documents
                    new_inst['negative_ctxs'] = []
                    
                    if mode in ['random', 'random+input', 'random+cont', 'random+cont+input', 'random+nv', 'random+nv+input']:
                        # get the random negative documents
                        for _ in range(5):
                            random_inst = random.choice(data)
                            while random_inst['question_text'] == inst['question_text']:
                                random_inst = random.choice(data)
                            new_inst['negative_ctxs'] += random_inst['ground_truths'][:5]
                        new_inst['hard_negative_ctxs'] = []
                    # elif mode in ['cont+input']:                    # put cont doc in the "negative" list
                    #     for ret in contriever_top_doc[:5]:
                    #         if ret['title'] not in new_ground_truth_titles:
                    #             new_inst['negative_ctxs'].append(ret)

                    if mode in ['random+input', 'random+cont+input', 'random+nv+input']:
                        # get the hard negative documents (input negatives)
                        new_inst['input_negative_ctxs'] = [doc for doc in retrieved_documents]
                    
                    if mode in ['random+cont', 'random+cont+input']:    # put contriever doc in the "hard negative" list
                        for ret in contriever_top_doc[:5]:
                            if ret['title'] not in new_ground_truth_titles:
                                new_inst['hard_negative_ctxs'].append(ret)
                    if mode in ['random+nv', 'random+nv+input']:
                        for ret in nv_top_doc[:5]:                      # put nv doc in the "hard negative" list
                            if ret['title'] not in new_ground_truth_titles:
                                new_inst['hard_negative_ctxs'].append(ret)
                    
                    i += 1
                    num_combo += 1
                    new_data.append(new_inst)
            
            print(len(new_data))
            len_eval_data = len(new_data) // 10
            new_eval_data = new_data[:len_eval_data]
            new_data = new_data[len_eval_data:]

            # write the new ground truth to a jsonl file
            write_jsonl(new_data, rootdir / f'training/data/{data_type}_data/{corpus}_corpus/{split}_{mode}_data_inp{TOP_N}.jsonl')
            write_jsonl(new_eval_data, rootdir / f'training/data/{data_type}_data/{corpus}_corpus/{split}_eval_{mode}_data_inp{TOP_N}.jsonl')


        # input - question + 3 positive ctxs
        # negative_gt - 3 positive ctxs
        # gt - original gt - 3 positive ctxs
        # also eval mrecall @ 5 for original gt. 


#### Get the dev data (Each instance in dev_gt_data must be mapped to one instance here)

In [ ]:

"""
    Generating test (dev) data for fine-tuning the 2nd stage model. 
    The test data consists of the following:
        - rewritten question: Question + top N ground truth documents (input documents)
        - positive documents: documents that are in the ground truth but not in the input documents
        - hard negative documents: documents that are in the input documents
        
    Also, check if the rewritten question does not exceed 512 tokens. Truncate the instance if it does.
"""
from pathlib import Path
split='dev'
corpus='qampari'
if split == 'dev':
    # # get top N retrieved documents
    rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/')

    # get ground truth documents
    data_path = rootdir / f'data/{data_type}_data/{split}_data_gt_{corpus}_corpus.jsonl'
    data = read_jsonl(data_path)

    # compute the difference 
    # identify the documents that are in the ground truth but not in the retrieved documents
    # these are the new ground truth
    for TOP_N in [0, 1, 2, 3]:
        new_data = []
        i = 0
        from tqdm import tqdm
        for inst in tqdm(data):
            if len(inst['ground_truths']) <= TOP_N:
                continue
            # get the input documents
            retrieved_documents = [doc[0] for doc in inst['ground_truths'][:TOP_N]]
            
            if TOP_N == 0:
                new_data.append({'id': i, 'question': inst['question_text'], 'org_q': inst['question_text']})
            else:
                # get the rewritten question based on the input documents
                rewritten_question = 'Question: [Question]\n\nDocuments: [Documents]'.replace('[Question]', inst['question_text']).replace('[Documents]', '\n'.join([ddd['title'] + ' ' + ddd['text'] for ddd in retrieved_documents]))  
                # check the rewritten question does not exceed 512 tokens
                if len(tokenizer(rewritten_question)['input_ids']) > 512:
                    toks = 0
                    print('=-===')
                    print('Exceed 512 tokens', len(tokenizer(rewritten_question)['input_ids']))
                    while len(tokenizer(rewritten_question)['input_ids']) > 512:
                        toks += 20
                        rewritten_question = 'Question: [Question]\n\nDocuments: [Documents]'.replace('[Question]', inst['question_text']).replace('[Documents]', '\n'.join([ddd['title'] + ' ' + ddd['text'][:-toks] for ddd in retrieved_documents]))  
                        print('reduced by 60 chars', len(tokenizer(rewritten_question)['input_ids']))
                
                new_data.append({'id': i, 'question': rewritten_question, 'org_q': inst['question_text']})
            # rewritten_question = 'Question: [Question]\n\nDocuments: [Documents]'.replace('[Question]', inst['question_text']).replace('[Documents]', '\n'.join([ddd['title'] + ' ' + ddd['text'] for ddd in retrieved_documents]))  
            # new_data.append({'id': i, 'question': rewritten_question, 'org_q': inst['question_text']})
            
            # get the positive documents
            new_data[-1]['positive_ctxs'] = [doc for doc in inst['ground_truths'][TOP_N:]]  
            
            # get the hard negative documents
            new_data[-1]['input_negative_ctxs'] = [doc for doc in retrieved_documents]
            
            i += 1
        print(len(new_data))

        # write the new ground truth to a jsonl file
        write_jsonl(new_data, rootdir / f'data/{data_type}_data/2nd_stage_test_data/{split}_data_{corpus}_corpus_inp{TOP_N}.jsonl')
    

### Check Input Negatives for 2nd Stage Training Data

In [8]:
# check if all data with input negatives with the field "input_negative_ctxs"
rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/')
split='train'
corpus='qampari'
_key = 'positive_ctxs'
from tqdm import tqdm

def check_input_negatives(file_path):
    data = read_jsonl(file_path)
    all_input_negatives = True
    empty = 0
    new_data = []
    for inst in tqdm(data):
        if _key not in inst:
            print('No input negatives')
            print(file_path)
            all_input_negatives = False
            break
        else:
            if len(inst[_key]) == 0:
                empty += 1
            else:
                new_data.append(inst)
    if all_input_negatives:
        print('All instances have input negatives')
        print('Empty input negatives', empty)
        # write_jsonl(new_data, file_path)
        
for mode in ['random', 'random+input', 'random+cont', 'random+cont+input', 'random+nv', 'random+nv+input']:
    # check_input_negatives(rootdir / f'training/data/{data_type}_data/{corpus}_corpus/{split}_{mode}_data.jsonl')
    check_input_negatives(rootdir / f'training/data/{data_type}_data/{corpus}_corpus/{split}_eval_{mode}_data.jsonl')
    

100%|██████████| 52312/52312 [00:00<00:00, 1773488.34it/s]


All instances have input negatives
Empty input negatives 0


100%|██████████| 52312/52312 [00:00<00:00, 1654357.19it/s]


All instances have input negatives
Empty input negatives 0


100%|██████████| 52312/52312 [00:00<00:00, 1327206.38it/s]


...

All instances have input negatives
Empty input negatives 0


100%|██████████| 52312/52312 [00:00<00:00, 1635343.71it/s]


All instances have input negatives
Empty input negatives 0


100%|██████████| 52312/52312 [00:00<00:00, 1351735.97it/s]


All instances have input negatives
Empty input negatives 0


### Add Original Question to the Training / Eval Set for Training Contriever

In [7]:
# check if all data with input negatives with the field "input_negative_ctxs"
rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/')
split='train'
corpus='qampari'
from tqdm import tqdm

def add_org_q(file_path):
    data = read_jsonl(file_path)

    for inst in tqdm(data):
        inst['org_question'] = inst['question'].split('\n\n')[0].split('Question: ')[1]
    write_jsonl(data, file_path)
        
for mode in ['random', 'random+input', 'random+cont', 'random+cont+input', 'random+nv', 'random+nv+input']:
    add_org_q(rootdir / f'training/data/{data_type}_data/{corpus}_corpus/{split}_{mode}_data.jsonl')
    add_org_q(rootdir / f'training/data/{data_type}_data/{corpus}_corpus/{split}_eval_{mode}_data.jsonl')

100%|██████████| 52312/52312 [00:00<00:00, 403041.25it/s]


KeyboardInterrupt: 

### Get input data with random negative / nv negatives / input negatives (For Training NV-Embed)

In [5]:
from pathlib import Path
import random
from tqdm import tqdm

rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/training/data/qampari_data/qampari_corpus/')
all_data = rootdir.glob('*_combined.jsonl')
all_data = list(all_data)
print(list(all_data))
out_path = Path('/scratch/cluster/hungting/projects/llm2vec/cache/diverse-data/')
# dict_keys(['id', 'question', 'org_q', 'positive_ctxs', 'negative_ctxs', 'hard_negative_ctxs'])
# input_negative_ctxs

new_data = []
for data_path in all_data:
    data = read_jsonl(data_path)
    is_eval = 'eval' in data_path.stem
    for inst in tqdm(data):
        if len(inst['positive_ctxs']) == 0:
            continue
        if is_eval:
            positives = [random.choice(inst['positive_ctxs'])]
        else:
            if len(inst['positive_ctxs']) > 3:
                positives = random.sample(inst['positive_ctxs'], 3)
            else:
                positives = inst['positive_ctxs']
        
        for pos in positives:
            pos_ctx = pos['title'] + ' ' + pos['text']
            # if 'input_negative_ctxs' in inst:  # additionally include input negatives
            #     neg = random.choice(inst['input_negative_ctxs'])
            #     neg_ctx = neg['title'] + ' ' + neg['text']
            #     new_data.append({
            #         "query": inst['question'],
            #         "positive": pos_ctx,
            #         "negative": neg_ctx,
            #     })
            if len(inst['hard_negative_ctxs']) > 0:
                neg = random.choice(inst['hard_negative_ctxs'])
                neg_ctx = neg['title'] + ' ' + neg['text']
                new_data.append({
                    "query": inst['question'],
                    "positive": pos_ctx,
                    "negative": neg_ctx,
                })
            if len(inst['hard_negative_ctxs']) == 0:  # if don't have hard negatives, use random negatives
                neg = random.choice(inst['negative_ctxs'])
                neg_ctx = neg['title'] + ' ' + neg['text']
                new_data.append({
                    "query": inst['question'],
                    "positive": pos_ctx,
                    "negative": neg_ctx,
                })
                
            
            

    write_jsonl(new_data, out_path / f'{data_path.stem}_triplet.jsonl')

[PosixPath('/scratch/cluster/hungting/projects/diverse_response/training/data/qampari_data/qampari_corpus/train_random+input_data_combined.jsonl'), PosixPath('/scratch/cluster/hungting/projects/diverse_response/training/data/qampari_data/qampari_corpus/train_eval_random+input_data_combined.jsonl')]


100%|██████████| 152279/152279 [00:10<00:00, 14176.88it/s]


#### Combining the 2nd stage retrieval data and original objective data

In [ ]:
"""
    Generating training data for fine-tuning the 2nd stage model. 
    In this cell, the generated training data contains half examples from the original training data \
        and half examples from the new training data.
        
    The new training data consists of the following:
        - rewritten question: Question + top N ground truth documents (input documents)
        - positive documents: documents that are in the ground truth but not in the input documents
        - negative documents: documents retrieved from contriever that are not in the "positive documents"
        - hard negative documents: documents that are in the input documents
        
    The original training data consists of the following:
        - rewritten question: Question 
        - positive documents: documents that are in the ground truth 
        - negative documents: documents retrieved from contriever that are not in the "positive documents"
        
    Also, check if the rewritten question does not exceed 512 tokens. Omit the instance if it does.
"""
split='train_combine'
import random
if split == 'train_combine':
    train_str = '_train_gt'

    # # get top N retrieved documents
    # TOP_N = 5
    rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/')

    # get ground truth documents
    data_path = rootdir / f'data/{data_type}_data/train_data_gt.jsonl'
    data = read_jsonl(data_path)

    # get contriever retrieved documents
    contriever_doc_path = rootdir / f'retrieval_outputs/{data_type}/{data_type}{train_str}_contriever.jsonl'
    contriever_top_docs = read_jsonl(contriever_doc_path)

    # compute the difference 
    # identify the documents that are in the ground truth but not in the retrieved documents
    # these are the new ground truth
    TOP_N = 3
    TOP_N_Negative = 5
    TOP_N_Negative_Candidate = 10
    new_data = []
    new_eval_data = []
    i = 0

    for inst, contriever_top_doc in zip(data, contriever_top_docs):
        if len(inst['ground_truths']) <= TOP_N:
            continue
        # get the input documents
        retrieved_documents = [doc for doc in inst['ground_truths'][:TOP_N]]
        rewritten_question = 'Question: [Question]\n\nDocuments: [Documents]'.replace('[Question]', inst['question_text']).replace('[Documents]', '\n'.join([ddd['title'] + ' ' + ddd['text'] for ddd in retrieved_documents]))  
        # # check the rewritten question does not exceed 512 tokens
        # if len(tokenizer(rewritten_question)['input_ids']) > 512:
        #     print('Exceed 512 tokens', len(tokenizer(rewritten_question)['input_ids']))
        #     continue
        
        new_data.append({'id': i, 'question': rewritten_question, 'org_q': inst['question_text']})

        # get the positive documents
        new_data[-1]['positive_ctxs'] = [doc for doc in inst['ground_truths'][TOP_N:]]
        new_ground_truth_titles = [doc['title'] for doc in inst['ground_truths'][TOP_N:]]    
        
        # get the negative documents
        new_data[-1]['negative_ctxs'] = []
        for ret in contriever_top_doc['ctxs'][:TOP_N_Negative_Candidate]:
            if ret['title'] not in new_ground_truth_titles:
                new_data[-1]['negative_ctxs'].append(ret)
        new_data[-1]['negative_ctxs'] = new_data[-1]['negative_ctxs'][:TOP_N_Negative]
        
        # get the hard negative documents
        new_data[-1]['hard_negative_ctxs'] = [doc for doc in inst['ground_truths'][:TOP_N]]
        
        i += 1
    
    print(len(new_data))
    len_eval_data = len(new_data) // 10
    new_eval_data = new_data[:len_eval_data]
    new_train_data = new_data[len_eval_data:]
    print(len(new_train_data), len(new_eval_data))  
    
    new_data = []
    for inst, contriever_top_doc in zip(data, contriever_top_docs):
        if len(inst['ground_truths']) <= TOP_N:
            continue
        # get the input documents
        retrieved_documents = [doc for doc in inst['ground_truths'][:TOP_N]]
        
        new_data.append({'id': i, 'question': inst['question_text'], 'org_q': inst['question_text']})

        # get the positive documents
        new_data[-1]['positive_ctxs'] = [doc for doc in inst['ground_truths']]
        new_ground_truth_titles = [doc['title'] for doc in inst['ground_truths']]  
        
        # get the negative documents
        new_data[-1]['negative_ctxs'] = []
        for ret in contriever_top_doc['ctxs'][:TOP_N_Negative_Candidate]:
            if ret['title'] not in new_ground_truth_titles:
                new_data[-1]['negative_ctxs'].append(ret)
        new_data[-1]['negative_ctxs'] = new_data[-1]['negative_ctxs'][:TOP_N_Negative]
                
        new_data[-1]['hard_negative_ctxs'] = []
        
        i += 1
    
    print(len(new_data))
    len_eval_data = len(new_data) // 10
    new_eval_data += new_data[:len_eval_data]
    new_train_data += new_data[len_eval_data:]
    print(len(new_train_data), len(new_eval_data))  
    random.shuffle(new_train_data)
    random.shuffle(new_eval_data)
    
    # write the new ground truth to a jsonl file
    write_jsonl(new_train_data, rootdir / f'training/data/{data_type}_data/{split}_data_comb.jsonl')
    write_jsonl(new_eval_data, rootdir / f'training/data/{data_type}_data/{split}_eval_data_comb.jsonl')

#### Get the original objective training data (hard negative + random negative)

In [ ]:
"""
    Generate training data for fine-tuning on original QAMPARI.
    Include hard negatives and random negatives.
"""


import random
split='train_org'
if split == 'train_org':
    train_str = '_train_gt' 

    # # get top N retrieved documents
    # TOP_N = 5
    rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/')

    # get ground truth documents
    data_path = rootdir / f'data/{data_type}_data/train_data_gt.jsonl'
    data = read_jsonl(data_path)

    # get contriever retrieved documents
    nv_doc_path = rootdir / f'retrieval_outputs/{data_type}/{data_type}{train_str}_nv.jsonl'
    nv_top_docs = read_jsonl(nv_doc_path)

    # compute the difference 
    # identify the documents that are in the ground truth but not in the retrieved documents
    # these are the new ground truth
    TOP_N = 3
    TOP_N_Negative = 5
    TOP_N_Negative_Candidate = 10
    new_data = []
    new_eval_data = []
    new_train_data = []
    i = 0

    
    new_data = []
    for inst, nv_top_doc in zip(data, nv_top_docs):
        if len(inst['ground_truths']) <= TOP_N:
            continue        
        new_data.append({'id': i, 'question': inst['question_text'], 'org_q': inst['question_text']})

        # get the positive documents
        new_data[-1]['positive_ctxs'] = [doc for doc in inst['ground_truths']]
        new_ground_truth_titles = [doc['title'] for doc in inst['ground_truths']]  
        
        # get the negative documents
        new_data[-1]['negative_ctxs'] = []
        for _ in range(5):
            random_ret_inst = random.choice(nv_top_docs)
            for ret in random_ret_inst['ctxs'][:TOP_N_Negative_Candidate]:
                if ret['title'] not in new_ground_truth_titles:
                    new_data[-1]['negative_ctxs'].append(ret)
                    
        # get the negative documents
        new_data[-1]['hard_negative_ctxs'] = []
        for ret in nv_top_doc['ctxs'][:TOP_N_Negative_Candidate]:
            if ret['title'] not in new_ground_truth_titles:
                new_data[-1]['hard_negative_ctxs'].append(ret)
        new_data[-1]['hard_negative_ctxs'] = new_data[-1]['hard_negative_ctxs'][:TOP_N_Negative]
        
        i += 1
    
    print(len(new_data))
    len_eval_data = len(new_data) // 10
    new_eval_data += new_data[:len_eval_data]
    new_train_data += new_data[len_eval_data:]
    print(len(new_train_data), len(new_eval_data))  
    random.shuffle(new_train_data)
    random.shuffle(new_eval_data)
    
    # write the new ground truth to a jsonl file
    write_jsonl(new_train_data, rootdir / f'training/data/{data_type}_data/{split}_data_org.jsonl')
    write_jsonl(new_eval_data, rootdir / f'training/data/{data_type}_data/{split}_eval_data_org.jsonl')

#### Get the original objective training data (only random negative)

In [ ]:

"""
    Generate training data for fine-tuning on original QAMPARI.
    Include only random negatives.
"""

import random
if split == 'train_org_random':
    train_str = '_train_gt' 

    # # get top N retrieved documents
    # TOP_N = 5
    rootdir = Path('/scratch/cluster/hungting/projects/diverse_response/')

    # get ground truth documents
    data_path = rootdir / f'data/{data_type}_data/train_data_gt.jsonl'
    data = read_jsonl(data_path)

    # get contriever retrieved documents
    nv_doc_path = rootdir / f'retrieval_outputs/{data_type}/{data_type}{train_str}_nv.jsonl'
    nv_top_docs = read_jsonl(nv_doc_path)

    # compute the difference 
    # identify the documents that are in the ground truth but not in the retrieved documents
    # these are the new ground truth
    TOP_N = 3
    TOP_N_Negative = 5
    TOP_N_Negative_Candidate = 10
    new_data = []
    new_eval_data = []
    new_train_data = []
    i = 0

    
    new_data = []
    for inst, nv_top_doc in zip(data, nv_top_docs):
        if len(inst['ground_truths']) <= TOP_N:
            continue        
        new_data.append({'id': i, 'question': inst['question_text'], 'org_q': inst['question_text']})

        # get the positive documents
        new_data[-1]['positive_ctxs'] = [doc for doc in inst['ground_truths']]
        new_ground_truth_titles = [doc['title'] for doc in inst['ground_truths']]  
        
        # get the negative documents
        new_data[-1]['negative_ctxs'] = []
        for _ in range(5):
            random_ret_inst = random.choice(nv_top_docs)
            for ret in random_ret_inst['ctxs'][:TOP_N_Negative_Candidate]:
                if ret['title'] not in new_ground_truth_titles:
                    new_data[-1]['negative_ctxs'].append(ret)
            # new_data[-1]['negative_ctxs'] += new_data[-1]['negative_ctxs'][:TOP_N_Negative]
                
        new_data[-1]['hard_negative_ctxs'] = []
        
        i += 1
    
    print(len(new_data))
    len_eval_data = len(new_data) // 10
    new_eval_data += new_data[:len_eval_data]
    new_train_data += new_data[len_eval_data:]
    print(len(new_train_data), len(new_eval_data))  
    random.shuffle(new_train_data)
    random.shuffle(new_eval_data)
    
    # write the new ground truth to a jsonl file
    write_jsonl(new_train_data, rootdir / f'training/data/{data_type}_data/{split}_data_random.jsonl')
    write_jsonl(new_eval_data, rootdir / f'training/data/{data_type}_data/{split}_eval_data_random.jsonl')